# ESGF CMIP6 quickstart

Search the full CMIP6 archive on the Earth System Grid Federation and download a file directly via HTTP. No registration, no API key, no queue.

**Prerequisites:**
- Python 3.10+ with `requests`, `xarray`, `netcdf4`, `matplotlib`, `cartopy`, `numpy`
- Internet access (the script queries the DKRZ index node and downloads from ESGF data nodes)
- No credentials needed: CMIP6 data is open access (CC BY 4.0)

**What this notebook does:**
1. Search ESGF for monthly near-surface temperature from MPI-ESM1-2-LR historical
2. Download the first matching file (~8 MB, 20 years of monthly data)
3. Open with xarray and inspect the structure
4. Plot a map and a time series

Full documentation: [docs/esgf-cmip6/README.md](../docs/esgf-cmip6/README.md)

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
SEARCH_NODE = "https://esgf-data.dkrz.de/esg-search/search"
SOURCE_ID = "MPI-ESM1-2-LR"
EXPERIMENT_ID = "historical"
VARIABLE_ID = "tas"
TABLE_ID = "Amon"
VARIANT_LABEL = "r1i1p1f1"
GRID_LABEL = "gn"
OUTPUT_DIR = "./data/esgf-cmip6"
# ==================================================================

## Imports

Standard scientific stack plus `requests` for the ESGF REST API. No ESGF-specific packages needed.

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import requests
import xarray as xr

# Find repo root so common/ imports work regardless of where Jupyter launched
_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "CLAUDE.md").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

print(f"requests     {version('requests')}")
print(f"xarray       {version('xarray')}")
print(f"netcdf4      {version('netcdf4')}")
print(f"matplotlib   {version('matplotlib')}")
print(f"numpy        {version('numpy')}")

## Search ESGF

Query the DKRZ index node for files matching our facets. The response is JSON with a `response.docs` array containing one record per file. Each record has a `url` field with pipe-separated triples: `download_url|mime_type|service_name`. We extract the `HTTPServer` URL for direct download.

In [ ]:
params = {
    "project": "CMIP6",
    "source_id": SOURCE_ID,
    "experiment_id": EXPERIMENT_ID,
    "variable_id": VARIABLE_ID,
    "table_id": TABLE_ID,
    "variant_label": VARIANT_LABEL,
    "grid_label": GRID_LABEL,
    "latest": "true",
    "replica": "false",
    "type": "File",
    "format": "application/solr+json",
    "limit": 20,
}

resp = requests.get(SEARCH_NODE, params=params, timeout=60)
resp.raise_for_status()
data = resp.json()

num_found = data["response"]["numFound"]
docs = data["response"]["docs"]
print(f"Total files on ESGF: {num_found}")
print(f"Returned in this page: {len(docs)}")

# Extract HTTP download URLs
http_urls = []
for doc in docs:
    for url_entry in doc.get("url", []):
        parts = url_entry.split("|")
        if len(parts) == 3 and parts[2] == "HTTPServer":
            http_urls.append(parts[0])
            break

print(f"\nFirst 3 files:")
for url in http_urls[:3]:
    print(f"  {url.split('/')[-1]}")

## Download

Download the first file. CMIP6 files on ESGF follow the Data Reference Syntax naming convention, so the filename already encodes the model, experiment, variable, member, and date range. Each file covers a chunk of the full time range (typically 20 years for monthly data).

In [ ]:
file_url = http_urls[0]
filename = file_url.split("/")[-1]
output_path = Path(OUTPUT_DIR) / filename
output_path.parent.mkdir(parents=True, exist_ok=True)

if output_path.exists() and output_path.stat().st_size > 0:
    print(f"Already downloaded: {output_path}")
else:
    print(f"Downloading {filename} ...")
    with requests.get(file_url, stream=True, timeout=600) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print(f"Saved to {output_path}")

print(f"Size: {output_path.stat().st_size / 1e6:.2f} MB")

## Open with xarray

CMIP6 files are NetCDF-4 with CF conventions. Note that CMIP6 dimensions are typically `lat`/`lon` (not `latitude`/`longitude` as in ERA5), and longitude runs 0-360 rather than -180 to 180. Temperature is in Kelvin.

In [ ]:
ds = xr.open_dataset(output_path)
print(ds)
print(f"\nVariable '{VARIABLE_ID}' shape: {ds[VARIABLE_ID].shape}")
print(f"Time range: {str(ds.time.values[0])[:10]} to {str(ds.time.values[-1])[:10]}")
print(f"Lat: {ds.lat.values.min():.1f} to {ds.lat.values.max():.1f}")
print(f"Lon: {ds.lon.values.min():.1f} to {ds.lon.values.max():.1f}")

## Plot

A map of the time-mean temperature field, and a global-mean time series. The map uses `cartopy` for coastlines. We convert longitude from 0-360 to -180/180 so the map centres on the Atlantic rather than the Pacific.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

da = ds[VARIABLE_ID]

# Shift longitude from 0..360 to -180..180
da = da.assign_coords(lon=(((da.lon + 180) % 360) - 180)).sortby("lon")

# Convert Kelvin to Celsius for readability
da_c = da - 273.15

# --- Map: time-mean ---
fig = plt.figure(figsize=(10, 5))
ax = plt.axes(projection=ccrs.PlateCarree())
da_c.mean("time").plot(
    ax=ax, transform=ccrs.PlateCarree(), cmap="RdYlBu_r",
    cbar_kwargs={"label": f"{VARIABLE_ID} (deg C)"},
)
ax.coastlines(resolution="110m", linewidth=0.6)
ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="gray")
ax.set_title(f"{SOURCE_ID} {EXPERIMENT_ID} - time-mean {VARIABLE_ID}")
plt.tight_layout()
plt.show()

# --- Time series: area-weighted global mean ---
weights = np.cos(np.deg2rad(da_c.lat))
weighted_mean = da_c.weighted(weights).mean(["lat", "lon"])

fig, ax = plt.subplots(figsize=(10, 3.5))
weighted_mean.plot(ax=ax)
ax.set_ylabel(f"{VARIABLE_ID} (deg C)")
ax.set_title(f"{SOURCE_ID} {EXPERIMENT_ID} - global mean {VARIABLE_ID}")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Next steps

- **More models:** Change `SOURCE_ID` to compare models. Popular choices: `CESM2`, `UKESM1-0-LL`, `EC-Earth3`, `IPSL-CM6A-LR`. Some models use `r1i1p1f2` as their flagship member (UKESM1-0-LL, CNRM-CM6-1, HadGEM3-GC31-LL).
- **SSP scenarios:** Change `EXPERIMENT_ID` to `ssp245`, `ssp585`, etc. for future projections (2015-2100).
- **Daily data:** Set `TABLE_ID = "day"` for daily frequency. Files will be larger.
- **Multiple ensemble members:** Search with a different `VARIANT_LABEL` (e.g. `r2i1p1f1`, `r3i1p1f1`) to sample internal variability. The app's ensemble explorer can plot spread and statistics across members.
- **Full time range:** The search returns multiple files covering different date chunks. Download all of them and use `xr.open_mfdataset()` to concatenate.
- **CDS route:** For a simpler interface with fewer options, see the [CDS-based CMIP6 page](../docs/cmip6/README.md) which uses the Copernicus CDS API.
- **Streaming alternative:** The [EDH catalogue explorer](../docs/earth-data-hub/README.md) hosts 6 CMIP6 models as Zarr stores for lazy streaming access.

Full documentation: [docs/esgf-cmip6/README.md](../docs/esgf-cmip6/README.md)